# SLU2 - Subsetting data in Pandas: Learning notebook

In this notebook we will cover the following topics: 

* Indexing
* Selecting columns
* Selecting rows
* Chain indexing (not good) vs Multi-axis indexing (good)
* Masks
* Where
* Subsetting on conditions
* Adding columns and rows
* Removing columns and rows
    
You will make use of the following basic pandas functions and methods:

* [`read_csv`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html)
* [`sort_index`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sort_index.html)
* [`reset_index`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.reset_index.html)
* [`set_index`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.set_index.html)
* [`iloc`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.iloc.html)
* [`loc`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.loc.html)
* [`mask`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.mask.html)
* [`where`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.where.html)
* [`isin`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.isin.html)
* [`assign`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.assign.html)
* [`drop`](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.drop.html)

<br>

Let's start by importing pandas, like we learned in the previous unit. It will be the only thing that we will need for this learning unit as well.

In [ ]:
import pandas as pd

# This is an option to preview less rows in the notebook's cells' outputs
pd.options.display.max_rows = 10

## Indexing  in pandas

An **index** is a labeling that allows us to locate data points across a Dataframe or series more easily. You can think of it as an address system for our data, or in a way similar to a system that a library or archive would have. There's a notion of what is the main characteristic that you use to organize your data (eg. book titles or genre) and everything is organized and located using those characteristics. 

![data indexing](media/looking_for_data.jpg)

In the same way in pandas we use labels to organize and track your data. Both rows and columns have these labels, and usually we refer loosely to the rows indices as **index** and the column indices as **column names**. Proper indexing is what provides us with efficient ways of finding data across our datasets.

**Indexing** is then the process of selecting particular rows and columns of data from a DataFrame, using this address system. For example, you might want to point at the 3rd row or the 2nd column, or specifically specify the column name or row element by this label. You'll see how to do all of these in this notebook.

<br>

### Dataframe index (row labelling)

Let's start by understanding how rows are labelled. We'll read the data that we'll use in this unit from the file `airbnb_input.csv`, which is located in the `data/` directory. For this, we'll use the function `read_csv`, which was already shown in the previous unit.

In [ ]:
# Read the data in file airbnb_input.csv into a pandas DataFrame
df = pd.read_csv('data/airbnb_input.csv')

# Preview the first rows of the DataFrame.
df.head()

Pandas automatically creates an index for the rows that ranges from 0 to the length of the DataFrame minus 1 - this is the leftmost column you see above. But what if we want to use another column as this labelling? For example ids, such as __room_id__ or __host_id__ might sound like good possibilities for the index. 

To do that, you can specify the column name you want to use as this set of labels to the `read_csv` function, by passing it as the `index_col` argument. Let's say we want to use the column __room_id__ as the DataFrame index:

In [ ]:
# Read the data in file airbnb_input.csv into a pandas DataFrame and use column room_id as the DataFrame index.
df = pd.read_csv('data/airbnb_input.csv', index_col='room_id')

# Preview the first rows of the DataFrame.
df.head()

### Sorting the index

With the `sort_index` function, we can sort the DataFrame along the index. 

In this particular case, our DataFrame df was already sorted along the index, according to the natural order of the type of data it has - in this case the ids are numbers, so they are sorted from smaller to bigger. However, we can change it from bigger to smaller ids, by using the __ascending=False__ parameter.

In [ ]:
# Original df - sorted from smaller to bigger
df.head()

In [ ]:
# df with the index sorted from bigger to smaller room_id
df.sort_index(ascending=False)

### Changing the index

We can also change the index of a DataFrame after it's loaded, with the functions `reset_index` and `set_index`. When we do this, we may choose to keep the old index as a column in the DataFrame or dropping it completely. This is done through the argument __drop__, available in both functions. 


#### Resetting the index

The function `reset_index` will reset your index to the default one from pandas - a range from 0 to the length of the DataFrame minus 1, as mentioned above. In this function, the default behavior is to not drop the index (`drop=False`). See below what happens when we run it on our dataframe, without dropping the previous index (__room_id__).

In [ ]:
# Resetting the index and keeping the old as a column (room_id) - default behavior
df.reset_index()

Now notice the argument __drop__ added and the missing column in the next dataframe.

In [ ]:
# Resetting the index and dropping it 
df.reset_index(drop=True)

#### Setting the index

The function `set_index` allows us to pick a new column as our index. This function always discards the old index, and it replaces it by this new column. By default, this column will be moved to the index, and as such not kept as a regular column (`drop=True`). However, we can override it by setting the argument to `False`. See below both examples:

In [ ]:
# Setting column neighborhood as the new index - default behavior moves the column to the index
df.set_index('neighborhood')

In [ ]:
# Setting column neighborhood as the new index - use `drop` argument to keep as a regular column besides the index
df.set_index('neighborhood', drop=False)

__Note__: Both functions to set and reset the index of the dataframe will generate a new dataframe with the desired index, they won't mutate your current dataframe. In all examples above we are just visualizing that output, and not assigning it to any new variable, so if we now look into our original dataframe, it still has the original index from `df = pd.read_csv('data/airbnb_input.csv', index_col='room_id')` - __room_id__, as demonstrated below.

In [ ]:
# Looking into index of a dataframe
df.index.name

<br>
    
## Selecting columns

As mentioned before, **indexing** is what we call the process of selecting particular rows and columns of data from a DataFrame. We'll start by seeing how we can make use of the columns' indices - the __column names__ - to perform this task. 

There are two ways of doing this:

* using dot notation (`dataframe.column_name`)
* using braket notation (`dataframe[column_name]`)

<br>

### Selecting columns by name - dot notation

Using __dot notation__, you can select __one column__ from your DataFrame, obtaining a __Series__ with the column values. Try this out by selecting the __room_type__ column using dot notation:

In [ ]:
df.room_type

### Selecting columns by name - brackets notation

Using __brackets__, we can select __one or more columns__ from the DataFrame. Try selecting the same column as before:

In [ ]:
df['room_type']

Now try selecting multiple columns, such as __room_type__ and __neighborhood__. Notice that the output of the multiple selection is now a __Dataframe__ itself, containing the index and both columns selected

In [ ]:
df[['room_type', 'neighborhood']]

<br>

## Selecting rows

We'll now use the rows labels - the dataframe index - to select rows. We will show you two ways of doing this:

* Selecting rows by index position (`iloc`)
* Selecting rows by index labels (`loc`)

### Selecting rows by index position - iloc

With the `iloc` function you can select specific rows from a DataFrame, by their position in the index. To do this, you specify one integer, a list of integers or a slice. All indices specified should be between 0 and Dataframe length minus 1 (remember that Python starts indexing with a 0)

See how you can select the first row with this method. Notice that the output produced is a __Series__.

In [ ]:
df.iloc[0]

Now use a list of indices to fetch multiple rows. Notice that the output produced is now a __Dataframe__.

In [ ]:
df.iloc[[0, 2, 4, 6]]

Now see how you can provide a slice - a range provided in the format `start_row`:`end_row` - to obtain multiple rows. When your slice is at the beggining of the dataframe (your `start_row` is 0) or at the end of the dataframe (your `end_row` is the length of your dataframe minus 1) then you can omit it. See below some examples of slicing

In [ ]:
# Get first 3 rows
df.iloc[:3]

In [ ]:
# Get all rows from row 13225 to the last row
df.iloc[13225:]

In [ ]:
# Get all rows in between 4211 to 7442
df.iloc[4211:7442]

Another possibility when slicing is to pick only part of the elements in the provided range, by a given __step__. This will mean that we only select one row every `step` rows. This notation follows:

`start_row:end_row:step`

See how this works by selecting the first 10 columns with a step of 2.

In [ ]:
df.iloc[0:10:2,:]

### Selecting rows by index name - loc

Selecting by position is useful but oftentimes we want to specify the label to use to find a given row. You can do this with function `loc`. It follows the same notation of `iloc` but the content you are providing is an actual label. 

Let's see how you can do this. Start by selecting the row with __room_id__ 29396. Notice that once again, when you select only one row, a __Series__ object is returned.

In [ ]:
df.loc[29396]

Also note that if you search for an index that doesn't exist, you'll get a KeyError:

In [ ]:
try:
    df.loc[100]
except KeyError as e:
    print('\033[91mError\033[0m: {} not found'.format(e))

Similarly to the `iloc` function, you can also pass lists or slices of labels. You should get __Dataframe__ objects back.

In [ ]:
# Retrieving rows from list of labels
df.loc[[29396,17031]]

We can also apply the slice operation using the loc function. It is important to sort the index before doing this operation.

In [ ]:
# Retrieving rows from slices
df.loc[50000:900000,:]

You might have noticed that both the position system and our labels are integers. If you are wondering if the label selection only works for integers, you would be wrong.

One cool thing about all of these methods is that we can even use them with an index that is not integer. For example if our index is a string we can select rows from one label or a list of labels. Lets set the __neighborhood__ column as index and select some rows from the labels "Alvalade", "Estrela" and "Bel?m"

In [ ]:
# Select rows by one label
df.set_index('neighborhood').sort_index().loc['Alvalade']

In [ ]:
# Select rows by a list of labels
df.set_index('neighborhood').sort_index().loc[['Alvalade', 'Bel?m', 'Estrela']]

This also works for slicing. The notion of what is "in between" the labels uses the natural ordering of the index. In the case of strings, for example, this would be the alphabetic order of the elements. See below the result of slicing for a string index:

In [ ]:
# Select rows betweem 'Alvalade' and 'Belem'
df.set_index('neighborhood').sort_index().loc['Alvalade':'Bel?m',:]

<br>

## Multi-axis indexing

Selecting by rows or by columns corresponds to indexing by only one axis (we normally depict rows as axis 0 and columns as axis 1). However, one nice thing about loc and iloc is that it allows for multi-axis indexing, this is, we can select columns and rows at the same time. 

Let's use the `iloc` to select based on the position of the rows and columns to pick the last five rows and the first 3 columns.

In [ ]:
df.iloc[-5:,:3]

Let's now use the `loc` to select based on the names of the rows and columns to pick the _neighborhood_ and _price_ of the rooms 29396 and 17031.

In [ ]:
df.loc[[29396,17031],['neighborhood','price']]

<br>

## Performance remarks


### Chain indexing vs Multi-axis indexing

Imagine you are asked to select the neighborhood of room 17031. When we want to select a specific value in a DataFrame, given the row and the column, we might be tempted to do __chain indexing__, this is, first selecting by one index and then by the other, as shown below.

In [ ]:
## Chain indexing
df['room_type'][55116]

However, if you measure performance for both chain indexing or multi indexing, you'll see there is a good reason to pick the second. Run both cells below to measure the time each process takes (we do this by using `%%time`, a command used to count the time that the code in one cell took to run)

In [ ]:
%%time

## Chain indexing
df['room_type'][55116]

In [ ]:
%%time

## Multi indexing
df.loc[55116, 'room_type']

But why?

When we select a row or column in a DataFrame using brackets, the Python bellow Pandas is calling the  `getitem` method to return the requested data. Well, when we chain two sets of brackets, as in the first example, we are calling the  `getitem` method twice! On the other hand, when we use loc to select a value given a row and column at the same time, Python is only calling the `getitem` method once. 

In a small dataset like this the times might not be as different, but the bigger your dataframe the bigger this problem can get, so keep it in mind when you need to select by multiple axis.

<br>


## Subsetting data using mask and where 

When we are doing analysis on data, we usually want to filter it or select it according to certain conditons. Pandas dataframes provide two inbuilt methods that are useful to filter data according to a condition or a set of conditions:

* `df.mask` - Replace value when condition is true
* `df.where` - Replace value when condition is false



### Masks

The `mask` function replaces values when the condition passed is `True`, so it can be used to "hide" rows given a condition. These rows will have all values replaced by NaN:

In [ ]:
df.mask(df.overall_satisfaction == 5.0)

You can leave only the non-hidden values by dropping the NaN rows. Pandas already provides a function for that, `dropna`. 

__Hint__: notice the argument `how=all` in the function. This means that only masked rows - rows where __all__ elements are set to `NaN` - are dropped. 

In [ ]:
df.mask(df.overall_satisfaction == 5.0).dropna(how='all')

## Where

The `where` function, on the other hand, can be used to hide the rows that __do not__ verify a certain condition, leaving only the rows that do verify it. The "hidden" rows will have all values replaced by NaN.

In [ ]:
df.where(df.overall_satisfaction == 5.0)

Once again, we can leave only non-hidden values by using `dropna`

In [ ]:
df.where(df.overall_satisfaction == 5.0).dropna(how='all')

Basically __mask__ and __where__ do the opposite of each other! One is used to choose data given a condition and the other one is used to hide data given a condition. Applying the same condition to both and dropping "hidden" rows, you'll get complementary sets of your data.

<br>

## Subsetting data on conditions

Besides relying on `mask` and `where`, we can also use the __bracket notation__ with conditions to subset data from the DataFrame. By doing this, we get a DataFrame that most likely has a different shape from the initial one, since it only returns the subset of its rows that satisfy the condition.

Let's subset the DataFrame to get all the rooms in the Alvalade neighborhood. Note the DataFrame shape!

__Note__:  this is different from what we saw in the mask/filter functions: those functions don't change the DataFame shape, instead, they just replace the values that we don't want with NaNs. 

In [ ]:
df[df.neighborhood == 'Alvalade']

As another example, we're selecting the rooms in Alvalade, that have more than 10 reviews.

__Note the parenthesis around each condition, they're required!__

In [ ]:
df[(df.neighborhood == 'Alvalade') & (df.reviews > 10)]

### Special conditions

There are different types of conditions you may want to provide, and it is useful to know some of the basics operators you can use to subset:

- values of a column are equal to a specific value: `==` works for any type
- values of a column are __not__ equal to a specific value: `!=` works for any type
- other basic operator on numeric values:
   - Greater than, less than: `>` or `<`
   - Greater than or equal to `>=`
   - Less than or equal to `<=`
- values of a column are in a list of values: `isin` method 
- negate conditions with `~`  

The first three are either represented above, or are small variations of it. Play around with those to see different outcomes. The last two, however, might seem new to you. Let's look a bit more into those.


#### Matching values in a list

Sometimes you don't want to test equality with one value, but instead you want to check for several possible values. Even though you could match with each of the values in your list, as shown below:

In [ ]:
# match listings in Alvalade, Belem and Estrela

df[(df.neighborhood == 'Alvalade') | (df.neighborhood == 'Bel?m') | (df.neighborhood == 'Estrela')]

It may seem a good idea for a list of 3 elements, but what if you had 100 or 1000 elements? You could still hack it this way, but it is not ideal. There must be a better way! 

And there is. The function `isin` allows you to test against all elements in a list, making the task easier for you. Check the example above using this function:

In [ ]:
# match listings in Alvalade, Belem and Estrela

df[df.neighborhood.isin(['Alvalade', 'Bel?m', 'Estrela'])]

#### Negating a condition

What if we wanted to say that a given value is not in the desired list? Or if we just wanted to negate the conditions that were given to sample the "excluded" data. There are many ways to do this, but a very easy one is to use the `~` operator behind your conditions.

See the following examples to understand how this works. Let's start with a simple one using only one condition:

In [ ]:
# Start by running for a given condition
df[df.neighborhood == 'Alvalade']

In [ ]:
# Negate the previous condition
df[~(df.neighborhood == 'Alvalade')]

Now see what you can do when you have multiple conditions. See below how to negate only one condition, a subset of conditions or all of the conditions provided:

In [ ]:
# Use a more complex set of conditions
df[df.neighborhood.isin(['Alvalade', 'Bel?m', 'Estrela']) & (df.reviews > 100) & (df.price > 30)]

In [ ]:
# Negate first condition
df[~(df.neighborhood.isin(['Alvalade', 'Bel?m', 'Estrela'])) & (df.reviews > 100) & (df.price > 30)]

In [ ]:
# Negate second condition
df[(df.neighborhood.isin(['Alvalade', 'Bel?m', 'Estrela'])) & ~(df.reviews > 100) & (df.price > 30)]

In [ ]:
# Negate second and third conditions  notice the extra parentheses surrounding the desired conditions to negate
df[(df.neighborhood.isin(['Alvalade', 'Bel?m', 'Estrela'])) & ~((df.reviews > 100) & (df.price > 30))]

In [ ]:
# Negate all conditions - notice the extra parentheses surrounding ALL conditions
df[~((df.neighborhood.isin(['Alvalade', 'Bel?m', 'Estrela'])) & (df.reviews > 100))]

### Combining conditions

In many cases you will want to sample according to several conditions. As you see above, you should wrap your conditions in between parentheses. There are two main ways of combining conditions:

* AND: if you want to make sure your data satisfies all conditions
* OR: if you want to make sure your data satisfies one of the conditions

Even though python has `and` and `or` as regular keywords, for pandas subsetting you should use `&` and `|`. When using the logical operators  `and` and `or` you are implicitly asking Python to convert the conditions to boolean values. Numpy arrays and thus pandas columns have no truth value. So we need to use the bitwise operators, which numpy makes use of to do element-wise operations, since unlike their logical counterparts, they can be overriden to return specific output values.

See the examples below

In [ ]:
# Proper usage of & for element wise AND
df[(df.neighborhood == 'Alvalade') & (df.reviews > 10)]

In [ ]:
# Proper usage of | for element wise OR
df[(df.neighborhood == 'Alvalade') | (df.reviews > 10)]

In [ ]:
# Wrong usage of "and" for element wise AND
try:
    df[(df.neighborhood == 'Alvalade') and (df.reviews > 10)]
except ValueError as e:
    print("\033[91mError\033[0m: {}".format(e))

In [ ]:
# Wrong usage of "or" for element wise OR
try:
    df[(df.neighborhood == 'Alvalade') or (df.reviews > 10)]
except ValueError as e:
    print("\033[91mError\033[0m: {}".format(e))

<br>

## Adding Rows & Columns

### Adding or replacing a row

We can use the `loc` indexing operation to add a new row in a dataframe with a specific index. If dataframe already has any row with that index name then this will replace the contents of that row. See below an example of adding a non existing label:

In [ ]:
new_df = df.copy()
original_size = len(new_df.index)

# Add new room to our dataframe with roomid=100
new_df.loc[100] = [123456,'Private room','Entrecampos',82,4.5,2,1,29.0]
final_size = len(new_df.index)

# Show our new room
new_df.loc[100]

In [ ]:
print('Size of dataframe before: {} \nSize of dataframe after: {}'.format(original_size, final_size))

Now see what happens if you do that for an existing room_id:

In [ ]:
new_df = df.copy()
original_size = len(new_df.index)

# Add new room to our dataframe with roomid=100
new_df.loc[72807] = [123456,'Private room','Entrecampos',82,4.5,2,1,29.0]
final_size = len(new_df.index)

# Show our new room
new_df.loc[72807]

In [ ]:
print('Size of dataframe before: {} \nSize of dataframe after: {}'.format(original_size, final_size))

We can also use `iloc` to __replace__ the row at a given index position:

In [ ]:
new_df = df.copy()
original_size = len(new_df.index)

new_df.iloc[-1] = [56787,'Private room','Avenidas Novas',82,4.5,2,1,29.0]
final_size = len(new_df.index)

# Show our new room
new_df.tail()

In [ ]:
print('Size of dataframe before: {} \nSize of dataframe after: {}'.format(original_size, final_size))

### Adding or replacing a column

In the same way we did for rows, we can add a column to a daframe using the same notations that we used to select them, i.e., dot notation, brackets notation, or loc operator. See first some examples on how you can replace a column in your dataframe, for example let's normalize the prices to thousands of dollars instead of dollars:

In [ ]:
# Normalizes the price from dollars to thousands of dollars
new_df = df.copy()
new_df['price'] = new_df.price/1000
new_df.head()

Now see how you can use the same notation to add completely new columns:

In [ ]:
# Creates a new column in the DataFrame (price_per_week), where each row is equal to the price * 7
new_df = df.copy()
new_df['price_per_week'] = new_df.price * 7
new_df.head()

There is also a cool function called `assign`. This function returns a new dataframe with all the columns plus the one that we creating.

In [ ]:
# Creates new columns in the DataFrame (people_per_bedroom, price_per_month), 
# where each row is equal to the value of the accommodates column divided by the bedrooms column
new_df = df.copy()
new_df = new_df.assign(
    people_per_bedroom = new_df['accommodates']/new_df['bedrooms'],
    price_per_month = new_df['price'] * 31
)

new_df.head()

<br>

## Removing Rows & Columns

Finally, you can also remove columns or rows from your dataset. In order to drop rows and columns from a DataFrame, we can use function `drop`. Here's how you drop a row:

In [ ]:
# This drops the row with index 17031. This is the same than doing drop(17031,axis=0)
df.drop(labels=17031)

In order to drop a column, we do the following:

In [ ]:
# This drops column neighborhood. This is the same than doing drop('neighborhood',axis=1)
df.drop(columns='neighborhood')

If we want to drop multiple rows (or columns), we can use lists:

In [ ]:
df.drop(labels=[6499, 17031])


That's a wrap, you now have all the tools to subset your dataframes and sample your datasets. Play around with these functions, and then continue to the example and exercise notebooks to practice what you've learned!